In [ ]:
import requests
import re
import json
import subprocess
import sys
from typing import Dict, Any
from Bio import SeqIO

class MyFastaParser:
    def __init__(self, file_name):
        self.filename = file_name

    def _get_uniprot(self, accession: str) -> requests.Response:
        url = f"https://rest.uniprot.org/uniprotkb/{accession}"
        params = {"format": "json"}
        try:
            response = requests.get(url, params=params)
            return response
        except requests.exceptions.RequestException as e:
            print(f"Error fetching UniProt ID {accession}: {e}")
            return None

    def _uniprot_parse_response(self, resp: requests.Response) -> Dict[str, Any]:
        if resp is None or resp.status_code != 200:
            return {}

        data = resp.json()
        try:
            parsed = {
                "organism": data.get("organism", {}).get("scientificName", "N/A"),
                "geneInfo": data.get("genes", []),
                "sequenceInfo": data.get("sequence", {}),
                "type": data.get("entryType", "protein")
            }
            return parsed
        except Exception as e:
            print(f"Parsing error for UniProt: {e}")
            return {}

    def _get_ensembl(self, id: str) -> requests.Response:
        url = f"https://rest.ensembl.org/lookup/id/{id}"
        headers = {"Content-Type": "application/json"}
        try:
            response = requests.get(url, headers=headers)
            return response
        except requests.exceptions.RequestException as e:
            print(f"Error fetching ENSEMBL ID {id}: {e}")
            return None

    def _ensembl_parse_response(self, resp: requests.Response) -> Dict[str, Any]:
        if resp is None or resp.status_code != 200:
            return {}

        data = resp.json()
        try:
            parsed = { 
                "object_type": data.get("object_type"),
                "species": data.get("species"),
                "db_type": data.get("db_type"),
                "biotype": data.get("biotype"),
                "display_name": data.get("display_name"),
                "description": data.get("description"),
                "source": data.get("source")
            }
            return parsed
        except Exception as e:
            print(f"Parsing error for ENSEMBL: {e}")
            return {}

    def _access_database(self, db_id: str, database: str, seq_description: str, seq_sequence: str) -> dict:
        result = {
            f"file_info_{db_id}": {
                "description": seq_description,
                "sequence": str(seq_sequence)
            }
        }
        
        if database == "uniprot":
            resp = self._get_uniprot(db_id)
            db_data = self._uniprot_parse_response(resp)
        else:
            resp = self._get_ensembl(db_id)
            db_data = self._ensembl_parse_response(resp)

        if db_data:
            result[f"database_info_{db_id}"] = db_data
        else:
            result["WARNING"] = {"No ID match found."}
            
        return result

    def seqkit_stats(self) -> dict:
        try:
            result = subprocess.run(
                ["seqkit", "stats", self.filename, "--tabular", "--all"],
                capture_output=True, text=True, check=True
            )
            
            lines = result.stdout.strip().split('\n')
            if len(lines) < 2:
                return {"error": "Unexpected seqkit output format"}
            
            headers = lines[0].split('\t')
            values = lines[1].split('\t')
            
            stat_info = dict(zip(headers, values))
            
            return {
                'fasta_seqkit_stat_info': stat_info,
                'fasta_type': stat_info.get('type', 'Unknown'),
                'fasta_num_seqs': int(stat_info.get('num_seqs', 0))
            }

        except subprocess.CalledProcessError as e:
            return {"error": e.stderr.strip()}
        except Exception as e:
            return {"error": str(e)}

    def biopython_parser(self, seqkit_result: dict) -> dict:
        if "error" in seqkit_result:
            return seqkit_result

        fasta_type = seqkit_result.get('fasta_type', '').lower()
        output = {}
        
        if fasta_type == 'protein':
            db_name = "uniprot"
            id_regex = r'([OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2})'
        else:
            db_name = "ensembl"
            id_regex = r'(ENS[A-Z]+[0-9]{11})'

        output["DB_name"] = db_name

        try:
            for record in SeqIO.parse(self.filename, "fasta"):
                match = re.search(id_regex, record.description)
                
                if match:
                    db_id = match.group(1)
                    db_entry = self._access_database(db_id, db_name, record.description, record.seq)
                    output.update(db_entry)
                else:
                    output[f"file_info_{record.id}"] = {
                        "description": record.description,
                        "sequence": str(record.seq)
                    }
                    output["WARNING"] = {"No ID match found."}

            return output
        except Exception as e:
            return {"error": f"Biopython parsing failed: {str(e)}"}

    def show_output(self, output, indent=0):
        for key, value in output.items():
            print('\t' * indent + str(key))
            if isinstance(value, dict):
                self.show_output(value, indent + 1)
            else:
                print('\t' * (indent + 1) + str(value))

In [4]:
parser = MyFastaParser('test_file.fasta')
stats = parser.seqkit_stats()
stats

{'fasta_seqkit_stat_info': {'file': 'test_file.fasta',
  'format': 'FASTA',
  'type': 'Protein',
  'num_seqs': '2',
  'sum_len': '456',
  'min_len': '29',
  'avg_len': '228.0',
  'max_len': '427',
  'Q1': '29',
  'Q2': '228',
  'Q3': '427',
  'sum_gap': '0',
  'N50': '427',
  'N50_num': '1',
  'Q20(%)': '0',
  'Q30(%)': '0',
  'AvgQual': '0.00',
  'GC(%)': '0.00',
  'sum_n': '0'},
 'fasta_type': 'Protein',
 'fasta_num_seqs': 2}

In [5]:
biopython = parser.biopython_parser(stats)

In [6]:
parser.show_output(biopython)

DB_name
	uniprot
file_info_P11473
	description
		sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
	sequence
		MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
database_info_P11473
	organism
		Homo sapiens
	geneInfo
		[{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312', 'source': 'HGNC', 'id': 'HGNC:12679'}], 'value': 'VDR'}, 'synonyms': [{'value': 'NR1I1'}]}]
	sequenceInfo
		value
			MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSS